In [1]:
from core.preprocess import LLM_MODEL, OLLAMA_URL
print("LLM_MODEL =", LLM_MODEL)
print("OLLAMA_URL =", OLLAMA_URL)

LLM_MODEL = llama3:8b
OLLAMA_URL = http://localhost:11434


# 1) Gibt es überflüssige Chunks?

## 1.a) --- chunking von ChromaDB mit det Metadata anzeigen:  ---

In [ ]:
from chromadb import PersistentClient
from core.preprocess import COLLECTION, DB_PATH

def show_chunks_metadata():
    client = PersistentClient(path=DB_PATH)
    print("Collections:", [c.name for c in client.list_collections()])

    col = client.get_collection(COLLECTION)

    # Prüfen, ob Collection leer ist
    print(f"Anzahl Chunks in DB: {col.count()} ")

    # Alle Chunks holen (limit wichtig!)
    res = col.get(include=["metadatas", "documents"], limit=99999)

    print("Anzahl geladene Chunks:", len(res["ids"]))

    for m, d in zip(res["metadatas"], res["documents"]):
        print(m.get("document_id"), "→", m.get("chunk_id"))
        print(d)

if __name__ == "__main__":
    show_chunks_metadata()


## 1.b) --- chunking von markdown-dateien mit det Metadata anzeigen:  ---


In [ ]:
from utils.chunking import chunk_documents
from langchain.schema import Document
from typing import List
from core.preprocess import FINAL_MD_PATH
from langchain_community.document_loaders import TextLoader

def load_docs() -> List[Document]:
    docs = []
    for md in sorted(FINAL_MD_PATH.glob("*.md")):
        docs.extend(TextLoader(str(md), encoding="utf-8").load())
    return docs

def chunk_mardown():
    chunks = chunk_documents(load_docs(), 512, 75)
    print(f"Anzahl Chunks von Markdown: {len(chunks)}\n")

    for c in chunks:
        print(c.metadata["document_id"], "→", c.metadata["chunk_id"])

if __name__ == "__main__":
    chunk_mardown()